# Goodreads Book Genre Trends — Database Creation

## Overview

This notebook creates and populates the SQLite relational database for the Goodreads book genre analysis.

The cleaned and validated datasets produced during the **Data Preparation** stage are loaded from individual Parquet files and used to construct the relational database.

The database is designed to reduce unnecessary duplication while preserving the relationships between books, authors, Goodreads genres, and the original source genre datasets.

## Database Design

The database consists of seven relational tables.

### Entity Tables

- **books** — stores one record for each unique book.
- **authors** — stores one record for each unique author.
- **genres** — stores one record for each unique Goodreads genre.
- **source_genres** — stores the original genre/category represented by each source dataset.

### Relationship Tables

- **book_authors** — connects books to authors.
- **book_genres** — connects books to Goodreads genres.
- **book_source_genres** — connects books to the original source datasets in which they appeared.

The relationship tables allow books to have multiple authors, multiple genres, and multiple source classifications without duplicating the core book information.

## Database Relationships

The primary relational structure is:

**books → book_authors → authors**

**books → book_genres → genres**

**books → book_source_genres → source_genres**

Primary keys uniquely identify records within each entity table, while foreign keys establish the relationships between tables.

## Objectives

This notebook will:

1. Load the prepared relational datasets.
2. Validate the prepared data before database insertion.
3. Create the SQLite database.
4. Create the relational database tables.
5. Define primary-key and foreign-key constraints.
6. Populate the entity tables.
7. Populate the relationship tables.
8. Validate database row counts.
9. Verify foreign-key integrity.
10. Inspect the resulting database schema.
11. Run test relational queries to confirm that the database relationships function correctly.
12. Close the database connection after successful creation and validation.

## Input Data

The database is populated using the prepared Parquet files created in `data_preparation.ipynb`.

These files are located in:

`Data/prepared`

The prepared datasets are:

- `books.parquet`
- `authors.parquet`
- `book_authors.parquet`
- `genres.parquet`
- `book_genres.parquet`
- `source_genres.parquet`
- `book_source_genres.parquet`

The database creation notebook does not repeat the data-cleaning process. Instead, it uses the validated datasets produced during the previous stage.

## Database Output

The resulting SQLite database is stored as:

`Data/goodreads_capstone.db`

This database will serve as the primary relational data source for the SQL analysis portion of the capstone project.

## 1. Load Prepared Datasets

The validated relational datasets created during the data-preparation stage are loaded from their individual Parquet files.

Each DataFrame corresponds to a table that will be created in the SQLite database.

Separating the data into these DataFrames allows the database to be populated according to the relational design established during the data-preparation stage.

In [1]:
from pathlib import Path
import pandas as pd

prepared_folder = Path("../Data/prepared")

books = pd.read_parquet(prepared_folder / "books.parquet")
authors = pd.read_parquet(prepared_folder / "authors.parquet")
book_authors = pd.read_parquet(prepared_folder / "book_authors.parquet")
genres = pd.read_parquet(prepared_folder / "genres.parquet")
book_genres = pd.read_parquet(prepared_folder / "book_genres.parquet")
source_genres = pd.read_parquet(prepared_folder / "source_genres.parquet")
book_source_genres = pd.read_parquet(
    prepared_folder / "book_source_genres.parquet"
)

## 2. Validate Prepared Datasets

Before creating the SQLite database, the prepared DataFrames are inspected to verify their size, structure, data types, and contents.

This provides a checkpoint between the data-preparation and database-creation stages and helps identify problems before the data is inserted into SQLite.

### 2.1 Dataset Record Counts

The number of records in each prepared entity and relationship DataFrame is displayed.

These counts establish the expected row counts that will later be compared against the corresponding SQLite tables after the database has been populated.

In [2]:
# =========================================================
# Validate Prepared Datasets
# =========================================================

print("=" * 60)
print("PREPARED DATASET SUMMARY")
print("=" * 60)

print(f"Books:                 {len(books):,}")
print(f"Authors:               {len(authors):,}")
print(f"Book-Author links:     {len(book_authors):,}")
print(f"Genres:                {len(genres):,}")
print(f"Book-Genre links:      {len(book_genres):,}")
print(f"Source Genres:         {len(source_genres):,}")
print(f"Book-Source links:     {len(book_source_genres):,}")

PREPARED DATASET SUMMARY
Books:                 1,487,805
Authors:               482,461
Book-Author links:     1,488,070
Genres:                1,433
Book-Genre links:      4,981,671
Source Genres:         100
Book-Source links:     4,291,574


### 2.2 Inspect Data Types

The data types of each prepared DataFrame are examined to verify that the columns contain the expected types before being written to SQLite.

This is particularly important for fields such as identifiers, publication years, ratings, and rating counts.

In [3]:
# Check the structure of each DataFrame

print("\nBOOKS")
print(books.dtypes)

print("\nAUTHORS")
print(authors.dtypes)

print("\nBOOK_AUTHORS")
print(book_authors.dtypes)

print("\nGENRES")
print(genres.dtypes)

print("\nBOOK_GENRES")
print(book_genres.dtypes)

print("\nSOURCE_GENRES")
print(source_genres.dtypes)

print("\nBOOK_SOURCE_GENRES")
print(book_source_genres.dtypes)


BOOKS
book_id          int64
book_key           str
name               str
pub_year         int16
star_rating    float64
num_ratings      int64
isbn_clean         str
dtype: object

AUTHORS
author_id      int64
author_name      str
dtype: object

BOOK_AUTHORS
book_id      int64
author_id    int64
dtype: object

GENRES
genre_id      int64
genre_name      str
dtype: object

BOOK_GENRES
book_id     int64
genre_id    int64
dtype: object

SOURCE_GENRES
source_genre_id       int64
source_genre       category
dtype: object

BOOK_SOURCE_GENRES
book_id            int64
source_genre_id    int64
dtype: object


### 2.3 Preview Prepared Data

The first records from each prepared DataFrame are displayed for a visual inspection.

This confirms that the prepared tables contain the expected columns and values before they are loaded into the relational database.

In [4]:
# Preview the prepared tables

display(books.head())
display(authors.head())
display(book_authors.head())
display(genres.head())
display(book_genres.head())
display(source_genres.head())
display(book_source_genres.head())

,book_id,book_key,name,pub_year,star_rating,num_ratings,isbn_clean
0,1,isbn:9780060932299,Sharpe's Devil,1992,4.14,8141,9780060932299
1,2,isbn:9780786836628,Blood Fever,2006,4.02,7516,9780786836628
2,3,isbn:9783770476374,Detektiv Conan vs. Kaito Kid,2004,4.33,231,9783770476374
3,4,isbn:9781772752014,Marvel's Captain America: Sub Rosa,2016,3.39,74,9781772752014
4,5,isbn:9780821714782,The Awakening,1984,3.87,305,9780821714782


,author_id,author_name
0,1,!
1,2,"""Albert"""
2,3,"""Big"" John McCarthy"
3,4,"""J"""
4,5,"""Janosch"""


,book_id,author_id
0,1,44881
1,2,71940
2,3,159536
3,4,100324
4,5,209437


,genre_id,genre_name
0,1,10th century
1,2,11th century
2,3,12th century
3,4,13th century
4,5,14th century


,book_id,genre_id
0,1,644
1,1,640
2,1,1370
3,1,36
4,1,878


,source_genre_id,source_genre
0,1,action
1,2,adult
2,3,adventure
3,4,amazon
4,5,american_history


,book_id,source_genre_id
0,1,1
1,2,1
2,3,1
3,4,1
4,5,1


## 3. Create SQLite Database

The SQLite database path is defined before establishing the database connection.

The database will be stored in the project's `Data` directory so that the completed relational database remains separate from the notebook files and prepared Parquet datasets.

In [5]:
# =========================================================
# Create SQLite Database
# =========================================================

import sqlite3

db_path = Path("../Data/goodreads_capstone.db")

print("Database location:")
print(db_path.resolve())

Database location:
C:\Users\sarah\Projects\Book_Genre_Trends_Analysis\Data\goodreads_capstone.db


## 4. Connect to SQLite Database

A connection to the SQLite database is established using Python's built-in `sqlite3` library.

SQLite foreign-key enforcement is explicitly enabled so that relationships between the database tables are validated according to the relational design.

Enabling foreign-key enforcement helps prevent relationship records from referencing nonexistent books, authors, genres, or source genres.

In [6]:
# =========================================================
# Connect to SQLite Database
# =========================================================

conn = sqlite3.connect(db_path)

# Enable foreign-key enforcement
conn.execute("PRAGMA foreign_keys = ON;")

print("Connected to SQLite database.")
print("Foreign keys enabled:", 
      conn.execute("PRAGMA foreign_keys;").fetchone()[0])

Connected to SQLite database.
Foreign keys enabled: 1


## 5. Create Database Tables

The seven tables required by the relational database design are created in SQLite.

The entity tables contain the primary records, while the relationship tables establish the many-to-many relationships between books and authors, books and genres, and books and source datasets.

Primary-key constraints ensure that each entity has a unique identifier. Foreign-key constraints ensure that relationship records reference valid records in their corresponding entity tables.

Composite primary keys are used in the relationship tables to prevent the same relationship from being inserted more than once.

In [7]:
# =========================================================
# Create Database Tables
# =========================================================

cursor = conn.cursor()

cursor.executescript("""
    
    -- =============================================
    -- BOOKS
    -- =============================================
    
    CREATE TABLE IF NOT EXISTS books (
        book_id INTEGER PRIMARY KEY,
        book_key TEXT NOT NULL UNIQUE,
        name TEXT NOT NULL,
        pub_year INTEGER,
        star_rating REAL,
        num_ratings INTEGER,
        isbn_clean TEXT
    );


    -- =============================================
    -- AUTHORS
    -- =============================================
    
    CREATE TABLE IF NOT EXISTS authors (
        author_id INTEGER PRIMARY KEY,
        author_name TEXT NOT NULL UNIQUE
    );


    -- =============================================
    -- GENRES
    -- =============================================
    
    CREATE TABLE IF NOT EXISTS genres (
        genre_id INTEGER PRIMARY KEY,
        genre_name TEXT NOT NULL UNIQUE
    );


    -- =============================================
    -- SOURCE GENRES
    -- =============================================
    
    CREATE TABLE IF NOT EXISTS source_genres (
        source_genre_id INTEGER PRIMARY KEY,
        source_genre TEXT NOT NULL UNIQUE
    );


    -- =============================================
    -- BOOK_AUTHORS
    -- =============================================
    
    CREATE TABLE IF NOT EXISTS book_authors (
        book_id INTEGER NOT NULL,
        author_id INTEGER NOT NULL,

        PRIMARY KEY (book_id, author_id),

        FOREIGN KEY (book_id)
            REFERENCES books(book_id)
            ON DELETE CASCADE,

        FOREIGN KEY (author_id)
            REFERENCES authors(author_id)
            ON DELETE CASCADE
    );


    -- =============================================
    -- BOOK_GENRES
    -- =============================================
    
    CREATE TABLE IF NOT EXISTS book_genres (
        book_id INTEGER NOT NULL,
        genre_id INTEGER NOT NULL,

        PRIMARY KEY (book_id, genre_id),

        FOREIGN KEY (book_id)
            REFERENCES books(book_id)
            ON DELETE CASCADE,

        FOREIGN KEY (genre_id)
            REFERENCES genres(genre_id)
            ON DELETE CASCADE
    );


    -- =============================================
    -- BOOK_SOURCE_GENRES
    -- =============================================
    
    CREATE TABLE IF NOT EXISTS book_source_genres (
        book_id INTEGER NOT NULL,
        source_genre_id INTEGER NOT NULL,

        PRIMARY KEY (book_id, source_genre_id),

        FOREIGN KEY (book_id)
            REFERENCES books(book_id)
            ON DELETE CASCADE,

        FOREIGN KEY (source_genre_id)
            REFERENCES source_genres(source_genre_id)
            ON DELETE CASCADE
    );

""")

conn.commit()

print("All database tables created successfully.")

All database tables created successfully.


## 6. Insert Entity Tables

The four entity DataFrames are inserted into their corresponding SQLite tables.

These tables contain the unique records for books, authors, Goodreads genres, and source genres.

The relationship tables are populated separately after the entity records have been inserted so that their foreign keys can reference existing primary-key records.

In [8]:
# =========================================================
# Insert Entity Tables
# =========================================================

books.to_sql(
    "books",
    conn,
    if_exists="append",
    index=False
)

authors.to_sql(
    "authors",
    conn,
    if_exists="append",
    index=False
)

genres.to_sql(
    "genres",
    conn,
    if_exists="append",
    index=False
)

source_genres.to_sql(
    "source_genres",
    conn,
    if_exists="append",
    index=False
)

conn.commit()

print("Entity tables populated successfully.")

Entity tables populated successfully.


## 7. Insert Relationship Tables

The three relationship DataFrames are inserted into their corresponding SQLite tables.

These tables connect the entity records through foreign keys:

- `book_authors` connects books and authors.
- `book_genres` connects books and Goodreads genres.
- `book_source_genres` connects books and the original source datasets.

This completes the relational structure of the SQLite database.

In [9]:
# =========================================================
# Insert Relationship Tables
# =========================================================

book_authors.to_sql(
    "book_authors",
    conn,
    if_exists="append",
    index=False
)

book_genres.to_sql(
    "book_genres",
    conn,
    if_exists="append",
    index=False
)

book_source_genres.to_sql(
    "book_source_genres",
    conn,
    if_exists="append",
    index=False
)

conn.commit()

print("Relationship tables populated successfully.")

Relationship tables populated successfully.


## 8. Validate Database Row Counts

The number of records in each SQLite table is counted and displayed.

These results are compared with the record counts from the prepared Pandas DataFrames to confirm that the expected data was successfully transferred into the database.

This provides a basic validation that no records were unintentionally lost during database creation.

In [10]:
# =========================================================
# Validate Database Row Counts
# =========================================================

tables = [
    "books",
    "authors",
    "book_authors",
    "genres",
    "book_genres",
    "source_genres",
    "book_source_genres"
]

print("=" * 60)
print("DATABASE ROW COUNTS")
print("=" * 60)

for table in tables:
    count = conn.execute(
        f"SELECT COUNT(*) FROM {table}"
    ).fetchone()[0]

    print(f"{table:25} {count:,}")

DATABASE ROW COUNTS
books                     1,487,805
authors                   482,461
book_authors              1,488,070
genres                    1,433
book_genres               4,981,671
source_genres             100
book_source_genres        4,291,574


## 9. Validate Foreign-Key Integrity

SQLite's foreign-key integrity is checked using `PRAGMA foreign_key_check`.

A successful check with no returned violations confirms that the relationship tables reference valid records in their corresponding parent tables.

This provides evidence that the relational database structure was created and populated correctly.

In [11]:
# =========================================================
# Foreign Key Integrity Check
# =========================================================

foreign_key_check = conn.execute(
    "PRAGMA foreign_key_check;"
).fetchall()

if len(foreign_key_check) == 0:
    print("Foreign-key integrity check PASSED.")
else:
    print("Foreign-key integrity check FAILED.")
    print(foreign_key_check)

Foreign-key integrity check PASSED.


## 10. Inspect Database Schema

The SQLite schema is inspected to confirm that the expected tables were created.

The database metadata is also examined for each table to verify its columns, primary keys, and overall structure.

This provides a final structural check that the implemented SQLite database matches the planned relational database design.

In [12]:
# =========================================================
# Inspect Database Schema
# =========================================================

schema = pd.read_sql_query("""
    SELECT
        name,
        type
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name;
""", conn)

display(schema)

,name,type
0,authors,table
1,book_authors,table
2,book_genres,table
3,book_source_genres,table
4,books,table
5,genres,table
6,source_genres,table


In [13]:
# Show columns and constraints for each table

for table in tables:
    print(f"\n{'=' * 60}")
    print(table.upper())
    print("=" * 60)

    table_info = pd.read_sql_query(
        f"PRAGMA table_info({table});",
        conn
    )

    display(table_info)


BOOKS


,cid,name,type,notnull,dflt_value,pk
0,0,book_id,INTEGER,0,None,1
1,1,book_key,TEXT,1,None,0
2,2,name,TEXT,1,None,0
3,3,pub_year,INTEGER,0,None,0
4,4,star_rating,REAL,0,None,0
5,5,num_ratings,INTEGER,0,None,0
6,6,isbn_clean,TEXT,0,None,0



AUTHORS


,cid,name,type,notnull,dflt_value,pk
0,0,author_id,INTEGER,0,None,1
1,1,author_name,TEXT,1,None,0



BOOK_AUTHORS


...

,cid,name,type,notnull,dflt_value,pk
0,0,book_id,INTEGER,1,None,1
1,1,genre_id,INTEGER,1,None,2



SOURCE_GENRES


,cid,name,type,notnull,dflt_value,pk
0,0,source_genre_id,INTEGER,0,None,1
1,1,source_genre,TEXT,1,None,0



BOOK_SOURCE_GENRES


,cid,name,type,notnull,dflt_value,pk
0,0,book_id,INTEGER,1,None,1
1,1,source_genre_id,INTEGER,1,None,2


## 11. Test Book-Author Relationships

A relational SQL query is used to join the `books`, `book_authors`, and `authors` tables.

This confirms that the database can successfully retrieve information about books and their associated authors through the foreign-key relationships.

The query also serves as an initial test of the relational structure that will be used for later SQL analysis.

In [14]:
# =========================================================
# Test Relational Query
# =========================================================

query = """
SELECT
    b.name AS book_title,
    a.author_name,
    b.pub_year,
    b.star_rating
FROM books AS b
JOIN book_authors AS ba
    ON b.book_id = ba.book_id
JOIN authors AS a
    ON ba.author_id = a.author_id
ORDER BY b.star_rating DESC
LIMIT 20;
"""

top_books = pd.read_sql_query(query, conn)

display(top_books)

,book_title,author_name,pub_year,star_rating
0,Journey Through Chaos: The Valley,Ward Williams,2015,5.0
1,In The Land Of Scarabs,Janna Yeshanova,2014,5.0
2,E'S 6,Satol Yuiga,2000,5.0
3,Eternal Requiem,J.S. Chancellor,2012,5.0
4,The Fairy Godmother Dilemma: Trollspell,Danyelle Leafty,2015,5.0
5,Political Punch: Contemporary Poems on the Pol...,Fox Frazier-Foley,2016,5.0
6,Boobs In Paradise,John David Lionel Brooke,2015,5.0
7,North of Khyber,Robert E. Howard,1987,5.0
8,"Labrador Wilderness, Newfoundland and Labrador...",Llewelyn Pritchard,2011,5.0
9,"Port Hope Simpson Mysteries Vol. 2, Newfoundla...",Llewelyn Pritchard,2011,5.0


## 12. Test Book-Genre Relationships

A second relational query joins the `books`, `book_genres`, and `genres` tables.

This confirms that individual books can be associated with their Goodreads genre classifications through the many-to-many relationship represented by `book_genres`.

This relationship will be particularly important for the capstone's analysis of genre trends over time.

In [15]:
# =========================================================
# Test Book-Genre Relationship
# =========================================================

query = """
SELECT
    b.name AS book_title,
    g.genre_name,
    b.pub_year,
    b.star_rating
FROM books AS b
JOIN book_genres AS bg
    ON b.book_id = bg.book_id
JOIN genres AS g
    ON bg.genre_id = g.genre_id
ORDER BY b.star_rating DESC
LIMIT 20;
"""

book_genres_test = pd.read_sql_query(query, conn)

display(book_genres_test)

,book_title,genre_name,pub_year,star_rating
0,Journey Through Chaos: The Valley,action,2015,5.0
1,In The Land Of Scarabs,contemporary romance,2014,5.0
2,In The Land Of Scarabs,action,2014,5.0
3,In The Land Of Scarabs,womens fiction,2014,5.0
4,E'S 6,manga,2000,5.0
5,E'S 6,action,2000,5.0
6,Eternal Requiem,adult,2012,5.0
7,Eternal Requiem,fantasy,2012,5.0
8,The Fairy Godmother Dilemma: Trollspell,adult,2015,5.0
9,Political Punch: Contemporary Poems on the Pol...,poetry,2016,5.0


## 13. Close Database Connection

After the database has been created, populated, and validated, the SQLite connection is closed.

Closing the connection ensures that all database operations have been completed and that the database file is no longer being held open by the notebook process.

In [16]:
# =========================================================
# Close Database Connection
# =========================================================

conn.close()

print("SQLite database connection closed.")
print(f"Database created at: {db_path.resolve()}")

SQLite database connection closed.
Database created at: C:\Users\sarah\Projects\Book_Genre_Trends_Analysis\Data\goodreads_capstone.db


## Database Creation Complete

The Goodreads relational database has been successfully created and validated.

The completed SQLite database contains separate entity and relationship tables for books, authors, Goodreads genres, and source datasets.

The database is now ready for the SQL analysis stage of the capstone, where intermediate and advanced queries can be used to investigate book publication and genre trends over time.